# Disney+ catalogue — Pandas practice (2 hours)

**Goal:** learn pandas by cleaning and analysing the *same historical Disney+ catalogue* used in `disney_cleaning_analysis.ipynb`. This is an **exercise notebook**, not a worked solution.

**Guiding question:** Among movies with at least **10,000 IMDb votes**, how do IMDb ratings differ by production-country group and release year?

| Exercise | Main skills | Target time |
|:--|:--|--:|
| 1. Load and explore | `read_csv`, `head`, `dtypes`, Series vs. DataFrame, sorting | 12 min |
| 2. Create useful columns | selection, `.copy()`, arithmetic, `mean(axis=1)`, rename/drop | 14 min |
| 3. Clean text, numbers and dates | `.str`, `to_numeric`, `to_datetime`, nullable integers | 18 min |
| 4. Inspect missingness and duplicates | `isna`, `duplicated`, `dropna` | 12 min |
| 5. Define movie samples | `apply`, `.loc`, boolean masks, `isin`, `between` | 18 min |
| 6. Summarise and compare | `groupby`, `.agg`, `size` vs `count`, `transform` | 18 min |
| 7. Reshape ratings | `melt`, `pivot`, `pivot_table` | 16 min |
| 8. Save and interpret | `to_csv`, `read_csv`, descriptive interpretation | 12 min |
| **Total** | | **120 min** |

### How to work

1. Run cells **in order**. Wherever you see `# TODO`, replace the placeholder with your code.
2. Use the **hints** only if you get stuck. You can consult the original Disney notebook, but try doing each step yourself first.
3. Run the **self-check** after finishing each exercise. These test key outcomes; passing a check does not necessarily mean your approach is the only correct one.
4. Answers are deliberately **not included**. Keep the original notebook separate to avoid accidentally seeing solutions.


> Important: IMDb scores use **0–10**, Metascores use **0–100**. These are descriptive comparisons in a selected catalogue, **not** population-level or causal conclusions.

## Setup — run this cell without changing it

This is supplied so the exercise is about **pandas**, not file-path troubleshooting. The loader checks the usual Week 4 locations. If your CSV is somewhere else, set `MANUAL_CSV_PATH` to its location (for Windows, a raw string is convenient).

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 25)

# Example override (leave as None if automatic search works):
# MANUAL_CSV_PATH = Path(r"C:\Users\Student\Downloads\disney_plus_shows.csv")
MANUAL_CSV_PATH = Path("disney_plus_shows.csv")

DATA_PATH = (
    Path(MANUAL_CSV_PATH).expanduser()
)
if DATA_PATH is None or not DATA_PATH.is_file():
    raise FileNotFoundError(
        "Disney CSV not found. Place disney_plus_shows.csv next to this notebook "
        "or set MANUAL_CSV_PATH in the setup cell to its full path. "
        "The attached Disney analysis notebook alone does not contain the data."
    )
DATA_PATH = DATA_PATH.resolve()
print("Found Disney catalogue:", DATA_PATH)
print("Working directory:", Path.cwd())

Found Disney catalogue: C:\Users\floba\OneDrive - Universität Zürich UZH\HS26\data_analysis_with_python\data-analysis-with-python-hs26\week_4\disney_plus_shows.csv
Working directory: c:\Users\floba\OneDrive - Universität Zürich UZH\HS26\data_analysis_with_python\data-analysis-with-python-hs26\week_4


---
## Exercise 1 — Load and explore (12 min)

### A. Import and basic inspection (about 6 min)

Read the Disney CSV into `df_raw`. Use `thousands=","`, like in the source notebook, so comma-formatted vote counts are handled correctly. Then fill in the three inspection variables.

- `shape`: tuple of rows and columns
- `column_types`: each column's dtype
- `type_counts`: counts of **all** values in `type`, including missing values

Use `display(...)` to inspect the first five rows. **Question:** Is one selected column a Series or a DataFrame?

In [11]:
df_raw = pd.read_csv(DATA_PATH, thousands=",")       # TODO: read DATA_PATH as a pandas DataFrame
shape = df_raw.shape         # TODO: rows and columns
column_types = df_raw.dtypes # TODO: pandas dtypes for every column
type_counts = df_raw["type"].value_counts(dropna=False) # TODO: value counts of "type", including missing values

# TODO: display the first 5 rows of df_raw
df_raw.head()

,imdb_id,title,plot,type,rated,year,released_at,added_at,runtime,genre,director,writer,actors,language,country,awards,metascore,imdb_rating,imdb_votes
0,tt0147800,10 Things I Hate About You,"A pretty, popular teenager can't go out on a d...",movie,PG-13,1999,31 Mar 1999,"November 12, 2019",97 min,"Comedy, Drama, Romance",Gil Junger,"Karen McCullah, Kirsten Smith","Heath Ledger, Julia Stiles, Joseph Gordon-Levi...","English, French",USA,2 wins & 13 nominations.,70.0,7.3,283945.0
1,tt7019028,101 Dalmatian Street,This series follows the lives of Delilah and D...,series,NaN,2018–,25 Mar 2019,"February 28, 2020",NaN,"Animation, Comedy, Family",NaN,NaN,"Josh Brener, Michaela Dietz, Bert Davis, Abiga...",English,"UK, USA, Canada",NaN,NaN,6.2,124.0
2,tt0115433,101 Dalmatians,An evil high-fashion designer plots to steal D...,movie,G,1996,27 Nov 1996,"November 12, 2019",103 min,"Adventure, Comedy, Crime, Family",Stephen Herek,"Dodie Smith (novel), John Hughes (screenplay)","Glenn Close, Jeff Daniels, Joely Richardson, J...","English, Spanish","USA, UK",Nominated for 1 Golden Globe. Another 3 wins &...,49.0,5.7,97785.0
3,tt0324941,101 Dalmatians 2: Patch's London Adventure,"Being one of 101 takes its toll on Patch, who ...",movie,G,2002,21 Jan 2003,"November 12, 2019",74 min,"Animation, Adventure, Comedy, Family, Musical","Jim Kammerud, Brian Smith","Jim Kammerud (story), Dan Root (story), Garret...","Barry Bostwick, Jason Alexander, Martin Short,...",English,USA,5 wins & 10 nominations.,NaN,5.8,7434.0
4,tt0211181,102 Dalmatians,Cruella DeVil gets out of prison and goes afte...,movie,G,2000,22 Nov 2000,"November 12, 2019",100 min,"Adventure, Comedy, Family",Kevin Lima,"Dodie Smith (novel), Kristen Buckley (story), ...","Glenn Close, Gérard Depardieu, Ioan Gruffudd, ...",English,"USA, UK",Nominated for 1 Oscar. Another 1 win & 7 nomin...,35.0,4.9,33444.0


### B. Select and sort (about 6 min)

Fill in these variables without modifying `df_raw`:

- `titles`: a **Series** containing just `title`
- `three_columns`: a **DataFrame** containing `title`, `year`, and `imdb_rating` (in that order)
- `sorted_movies_preview`: the first **10 rows** of `three_columns` sorted by `imdb_rating` **descending**

**Hint:** Selecting `df["col"]` and `df[["col"]]` gives different types. For descending order, use `ascending=False`.

In [9]:
titles = df_raw["title"]                 # TODO
three_columns = df_raw[["title", "year", "imdb_rating"]]          # TODO
sorted_movies_preview = three_columns.head(10).sort_values(by="imdb_rating", ascending=False)  # TODO

# TODO: display sorted_movies_preview
display(sorted_movies_preview)

,title,year,imdb_rating
8,A Celebration of the Music from Coco,2020,7.6
0,10 Things I Hate About You,1999,7.3
7,A Bug's Life,1998,7.2
6,"20,000 Leagues Under the Sea",1954,7.2
9,A Goofy Movie,1995,6.8
5,12 Dates of Christmas,2011,6.3
1,101 Dalmatian Street,2018–,6.2
3,101 Dalmatians 2: Patch's London Adventure,2002,5.8
2,101 Dalmatians,1996,5.7
4,102 Dalmatians,2000,4.9


**Self-check — Exercise 1**

In [12]:
assert isinstance(df_raw, pd.DataFrame)
assert {"type", "title", "year", "imdb_rating", "country", "imdb_votes"}.issubset(df_raw.columns)
assert shape == df_raw.shape
assert column_types.equals(df_raw.dtypes)
assert type_counts.sum() == len(df_raw)
assert isinstance(titles, pd.Series) and titles.name == "title"
assert isinstance(three_columns, pd.DataFrame)
assert list(three_columns.columns) == ["title", "year", "imdb_rating"]
assert isinstance(sorted_movies_preview, pd.DataFrame)
assert len(sorted_movies_preview) == min(10, len(df_raw))
assert sorted_movies_preview["imdb_rating"].is_monotonic_decreasing or (
    sorted_movies_preview["imdb_rating"].dropna().is_monotonic_decreasing
)
print("Exercise 1 passed!")

Exercise 1 passed!


---
## Exercise 2 — Select, calculate and rename (14 min)

### A. Select the core variables (3 min)

Create `df_work` by selecting **only** the columns below from `df_raw`. Keep the raw data unchanged by using `.copy()`.

In [14]:
core_columns = [
    "imdb_id", "type", "title", "director", "language", "country", "year",
    "released_at", "runtime", "imdb_rating", "metascore", "imdb_votes", "rated",
]

df_work = df_raw[core_columns].copy()  # TODO: choose core_columns and make an independent copy

### B. Put scores on the same scale (8 min)

Add four columns to `df_work`:

1. `imdb_pct`: IMDb rating on a **0–1** scale.
2. `metascore_pct`: Metascore on a **0–1** scale.
3. `avg_score_both`: arithmetic average of those two scores. If one is missing, the result should be missing.
4. `avg_score_available`: row-wise average of the two scores, **allowing one to be missing**.

**Think:** Why can `avg_score_available` exist for a movie where `avg_score_both` is missing?

**Hint:** `df[["column_a", "column_b"]].mean(axis=1)` calculates one mean per row. Arithmetic with `NaN` normally produces `NaN`.

In [17]:
# TODO: create these columns on df_work
df_work["imdb_pct"] = df_work["imdb_rating"] / 10
df_work["metascore_pct"] = df_work["metascore"] / 100
df_work["avg_score_both"] = df_work[["imdb_pct", "metascore_pct"]].mean(axis=1, skipna=False)
df_work["avg_score_available"] = df_work[["imdb_pct", "metascore_pct"]].mean(axis=1)

# TODO: display the first 5 rows of these four columns plus "title"
df_work[["title", "imdb_pct", "metascore_pct", "avg_score_both", "avg_score_available"]].head(5)

,title,imdb_pct,metascore_pct,avg_score_both,avg_score_available
0,10 Things I Hate About You,0.73,0.70,0.715,0.715
1,101 Dalmatian Street,0.62,NaN,NaN,0.620
2,101 Dalmatians,0.57,0.49,0.530,0.530
3,101 Dalmatians 2: Patch's London Adventure,0.58,NaN,NaN,0.580
4,102 Dalmatians,0.49,0.35,0.420,0.420


### C. Rename and remove the helper columns (3 min)

Rename `avg_score_both` to `avg_score_imdb_meta`. Afterwards, drop `imdb_pct` and `metascore_pct`, but keep both average columns.

**Hint:** `.rename(columns={"old": "new"})` and `.drop(columns=[...])` return new DataFrames unless you assign the result.

In [18]:
df_work = df_work.rename(columns={"avg_score_both": "avg_score_imdb_meta"})  # TODO: rename "avg_score_both" to "avg_score_imdb_meta"
df_work = df_work.drop(columns=["imdb_pct", "metascore_pct"])  # TODO: drop "imdb_pct" and "metascore_pct"

**Self-check — Exercise 2**

In [19]:
assert isinstance(df_work, pd.DataFrame) and df_work is not df_raw
assert set(core_columns).issubset(df_work.columns)
assert "avg_score_imdb_meta" in df_work and "avg_score_available" in df_work
assert "avg_score_both" not in df_work
assert "imdb_pct" not in df_work and "metascore_pct" not in df_work
expected_both = (df_raw["imdb_rating"] / 10 + df_raw["metascore"] / 100) / 2
expected_available = pd.concat(
    [df_raw["imdb_rating"] / 10, df_raw["metascore"] / 100], axis=1
).mean(axis=1)
assert np.allclose(df_work["avg_score_imdb_meta"], expected_both, equal_nan=True)
assert np.allclose(df_work["avg_score_available"], expected_available, equal_nan=True)
assert list(df_raw.columns) != [] and "avg_score_available" not in df_raw.columns
print("Exercise 2 passed!")

Exercise 2 passed!


---
## Exercise 3 — Clean strings, numbers and dates (18 min)

Keep working on a **copy** called `df_clean`.

### A. Convert the important types (12 min)

The original catalogue contains formats like `"1999"` or year ranges, `"97 min"`, and release dates such as `"31 Mar 1999"`.

- `year`: take its **first four characters**, convert to a number, then to pandas nullable integer type **`Int64`**.
- `imdb_votes`: convert to a number, then `Int64`.
- `runtime`: extract the first number of minutes from text, convert to `Int64`.
- `released_at`: convert to datetime with day-month-year format **`"%d %b %Y"`**.

**Hints:** `.str[:4]`, `.str.extract(r"(\d+)")[0]`, `pd.to_numeric(...)`, `.astype("Int64")`, and `pd.to_datetime(..., format=...)`. The original notebook uses `errors="raise"`; do the same so unexpected formats are visible.

You may want to inspect `df_raw[["year", "runtime", "released_at"]].head()` before converting anything.

In [20]:
df_raw[["year", "runtime", "released_at"]].head()

,year,runtime,released_at
0,1999,97 min,31 Mar 1999
1,2018–,NaN,25 Mar 2019
2,1996,103 min,27 Nov 1996
3,2002,74 min,21 Jan 2003
4,2000,100 min,22 Nov 2000


In [21]:
df_clean = df_work.copy()

# TODO: replace the columns with cleaned versions (keep missing values as missing)
df_clean["year"] = pd.to_numeric(df_clean["year"].str[:4]).astype("Int64")
df_clean["imdb_votes"] = pd.to_numeric(df_clean["imdb_votes"]).astype("Int64")
df_clean["runtime"] = df_clean["runtime"].str.extract(r"(\d+)")[0].astype("Int64")
df_clean["released_at"] = pd.to_datetime(df_clean["released_at"], format="%d %b %Y")

# TODO: show the columns title, year, imdb_votes, runtime, released_at
df_clean[["title", "year", "imdb_votes", "runtime", "released_at"]]

,title,year,imdb_votes,runtime,released_at
0,10 Things I Hate About You,1999,283945,97,1999-03-31
1,101 Dalmatian Street,2018,124,<NA>,2019-03-25
2,101 Dalmatians,1996,97785,103,1996-11-27
3,101 Dalmatians 2: Patch's London Adventure,2002,7434,74,2003-01-21
4,102 Dalmatians,2000,33444,100,2000-11-22
...,...,...,...,...,...
987,Zapped,2014,3750,102,2014-06-23
988,Zenon: Girl of the 21st Century,1999,6378,97,1999-01-23
989,Zenon: The Zequel,2001,3318,100,2001-01-12
990,Zenon: Z3,2004,2339,81,2004-06-11


### B. Derive a decade and search titles (6 min)

1. `release_decade`: from the numeric `year`, assign e.g. **1990** to 1999. Preserve missing years. Use integer arithmetic.
2. `star_wars_titles`: a DataFrame of rows whose `title` contains `"Star Wars"` (case-insensitive), showing `title`, `year` and `imdb_rating`. Make sure missing titles do not break the mask.

**Hint:** For the decade you can use integer division `// 10`, then multiplication by 10. For text filtering look at `.str.contains(..., case=False, na=False)` and `.loc[...]`.

In [24]:
df_clean["release_decade"] = df_clean["year"] // 10 * 10  # TODO
star_wars_titles = df_clean.loc[df_clean["title"].str.contains("Star Wars", case=False, na=False), ["title", "year", "imdb_rating"]]  # TODO

# TODO: display star_wars_titles.head(10)
star_wars_titles.head(10)

,title,year,imdb_rating
185,Phineas and Ferb: Star Wars,2014,8.2
260,Empire of Dreams: The Story of the 'Star Wars'...,2004,8.3
430,Lego Star Wars: All-Stars,2018,6.5
432,Lego Star Wars: Droid Tales,2015,7.5
433,LEGO Star Wars: The Resistance Rises,2016,5.6
434,Lego Star Wars: The Freemaker Adventures,2016,7.5
440,Lego Star Wars: The Yoda Chronicles,2013,7.5
658,Rogue One: A Star Wars Story,2016,7.8
708,Star Wars: Forces of Destiny,2017,5.0
709,Star Wars Rebels,2014,8.0


**Self-check — Exercise 3**

In [25]:
assert isinstance(df_clean, pd.DataFrame) and df_clean is not df_work
assert str(df_clean["year"].dtype) == "Int64"
assert str(df_clean["imdb_votes"].dtype) == "Int64"
assert str(df_clean["runtime"].dtype) == "Int64"
assert pd.api.types.is_datetime64_any_dtype(df_clean["released_at"])
assert len(df_clean) == len(df_raw)
assert df_clean["year"].isna().sum() == df_raw["year"].isna().sum()
assert df_clean["runtime"].isna().sum() == df_raw["runtime"].isna().sum()
assert df_clean["release_decade"].notna().sum() == df_clean["year"].notna().sum()
assert (df_clean["release_decade"].dropna() % 10 == 0).all()
assert (df_clean["year"].dropna() - df_clean["release_decade"].dropna()).between(0, 9).all()
assert isinstance(star_wars_titles, pd.DataFrame)
assert list(star_wars_titles.columns) == ["title", "year", "imdb_rating"]
assert star_wars_titles["title"].str.contains("Star Wars", case=False, na=False).all()
assert len(star_wars_titles) == df_raw["title"].str.contains("Star Wars", case=False, na=False).sum()
print("Exercise 3 passed!")

Exercise 3 passed!


---
## Exercise 4 — Missing values and duplicates (12 min)

### A. How much is missing? (6 min)

- `missing_count`: number of missing entries per column, sorted from most to least missing.
- `missing_share_pct`: missing **percentage** per column, rounded to **one decimal place**.
- `with_rating`: a view or copy of rows where IMDb rating is available, created with `dropna(subset=[...])`.

**Think:** Why would dropping every row containing *any* missing value be problematic here?

In [36]:
missing_count = df_clean.isna().sum().sort_values(ascending=False)   # TODO
missing_share_pct = round(df_clean.isna().mean() * 100, 1)  # TODO
with_rating = df_clean.dropna(subset=["imdb_rating"])         # TODO

# TODO: display the 8 columns with the most missing values
missing_count.head(8)

avg_score_imdb_meta    700
metascore              700
director               303
rated                  250
runtime                154
language               136
country                123
released_at            118
dtype: int64

### B. Check duplicates carefully (6 min)

Create a boolean Series `duplicate_id_mask` identifying rows with **duplicated, non-missing** IMDb IDs. Then build `duplicate_id_rows` containing those rows, sorted by `imdb_id`.

**Important:** `imdb_id` being missing does **not** mean that many different movies are duplicates.

**Hint:** Combine `.notna()` with `.duplicated(keep=False)` using `&`.

In [41]:
duplicate_id_mask = df_clean["imdb_id"].notna() & df_clean["imdb_id"].duplicated(keep=False)  # TODO: boolean Series
# TODO: select duplicate_id_rows with columns imdb_id, title, type, and sort by imdb_id
duplicate_id_rows = df_clean.loc[duplicate_id_mask, ["imdb_id", "title", "type"]].sort_values("imdb_id")

**Self-check — Exercise 4**

In [43]:
assert isinstance(missing_count, pd.Series)
assert missing_count["imdb_rating"] == df_clean["imdb_rating"].isna().sum()
assert missing_count.is_monotonic_decreasing
assert isinstance(missing_share_pct, pd.Series)
assert np.isclose(missing_share_pct["imdb_rating"],
                  round(df_clean["imdb_rating"].isna().mean() * 100, 1))
assert len(with_rating) == df_clean["imdb_rating"].notna().sum()
assert with_rating["imdb_rating"].notna().all()
assert isinstance(duplicate_id_mask, pd.Series)
assert duplicate_id_mask.dtype == bool
assert not duplicate_id_mask[df_clean["imdb_id"].isna()].any()
assert duplicate_id_mask.sum() == (df_clean["imdb_id"].notna() &
                                  df_clean["imdb_id"].duplicated(keep=False)).sum()
assert isinstance(duplicate_id_rows, pd.DataFrame)
assert len(duplicate_id_rows) == int(duplicate_id_mask.sum())
assert list(duplicate_id_rows.columns) == ["imdb_id", "title", "type"]
print("Exercise 4 passed!")

Exercise 4 passed!


---
## Exercise 5 — Groups and filtering (18 min)

### A. Country labels (6 min)

The same movie can have multiple production countries, separated by commas. Below is a **provided helper** from the logic of the source notebook (you don't have to write it). It assigns:

- `"usa"`: only USA-listed countries
- `"usa_other"`: USA **and** at least one non-USA country
- `"other"`: countries listed, but no USA
- missing (`pd.NA`): no country information

Apply `classify_country` to the `country` Series with `.apply(...)` to create `country_group` on `df_clean`. Convert the result to pandas `"string"` dtype.

In [53]:
USA_ALIASES = {"usa", "u.s.a.", "united states", "united states of america"}

def classify_country(value):
    if pd.isna(value) or not str(value).strip():
        return pd.NA
    countries = {part.strip().lower() for part in value.split(",") if part.strip()}
    has_usa = bool(countries & USA_ALIASES)
    has_other = bool(countries - USA_ALIASES)
    if has_usa and has_other:
        return "usa_other"
    if has_usa:
        return "usa"
    return "other"

# Try the helper on a few individual values first
print(classify_country("USA"))
print(classify_country("USA, UK"))
print(classify_country("France"))

df_clean["country_group"] = df_clean["country"].apply(classify_country).astype("string")  # TODO: apply classify_country and cast to "string"

usa
usa_other
other


### B. Define the analysis sample (7 min)

- `df_movies`: **all movies** in the cleaned catalogue (`type == "movie"`). Use `.loc` and `.copy()`.
- `df_analysis`: only those movies with **at least 10,000 IMDb votes**. Again, use `.loc` and `.copy()`.

Afterwards we supply a unique `movie_row_id` to simplify the later reshaping exercise. This is a unique row identifier, **not** an IMDb ID.

In [54]:
minimum_votes = 10_000

df_movies = df_clean.loc[df_clean["type"] == "movie"].copy()    # TODO: only movies
df_analysis = df_movies.loc[
    df_movies["imdb_votes"] >= minimum_votes
].copy()  # TODO: only movies with enough votes

# Run this AFTER your two TODO lines work:
# df_analysis["movie_row_id"] = df_analysis.index
df_analysis["movie_row_id"] = df_analysis.index


# TODO: print how many movie rows were kept before and after the vote filter
print(len(df_movies))
print(len(df_analysis))

680
280


### C. Multiple filtering conditions (5 min)

From `df_movies` select rows where the IMDb rating lies **between 6 and 8 inclusive** AND the content rating (`rated`) is either `"PG"` or `"PG-13"`.

Save the highest-rated **10** rows (columns `title`, `year`, `rated`, `imdb_rating`) as `family_preview`, ordered by rating descending.

**Hints:** `.between(6, 8)`, `.isin(["PG", "PG-13"])`, `&` for combining conditions, `.loc[mask, columns]`, and `.sort_values(...)`.

In [57]:
family_preview = df_movies.loc[(df_movies["imdb_rating"] >= 6) & (df_movies["imdb_rating"] <= 8) & ((df_movies["rated"] == "PG") | (df_movies["rated"] == "PG-13")), ["title", "year", "rated", "imdb_rating"]].sort_values("imdb_rating", ascending=False).head(10)  # TODO

# TODO: display family_preview
family_preview

,title,year,rated,imdb_rating
612,Pirates of the Caribbean: The Curse of the Bla...,2003,PG-13,8.0
991,Zootopia,2016,PG,8.0
912,The Nightmare Before Christmas,1993,PG,8.0
811,The Incredibles,2004,PG,8.0
498,The Avengers,2012,PG-13,8.0
418,Kitbull,2019,PG,8.0
493,Guardians of the Galaxy,2014,PG-13,8.0
149,Destino,2003,PG,7.9
495,Iron Man,2008,PG-13,7.9
500,Thor: Ragnarok,2017,PG-13,7.9


**Self-check — Exercise 5**

In [58]:
assert classify_country("USA, UK") == "usa_other"
assert classify_country("France") == "other"
assert pd.isna(classify_country(None))
assert set(df_clean["country_group"].dropna().unique()).issubset({"usa", "usa_other", "other"})
assert str(df_clean["country_group"].dtype) == "string"
assert df_clean["country_group"].notna().sum() == (
    df_clean["country"].notna() & df_clean["country"].fillna("").str.strip().ne("")
).sum()
assert isinstance(df_movies, pd.DataFrame) and isinstance(df_analysis, pd.DataFrame)
assert df_movies is not df_clean and df_analysis is not df_movies
assert df_movies["type"].eq("movie").all()
assert len(df_movies) == int(df_clean["type"].eq("movie").sum())
assert df_analysis["imdb_votes"].ge(minimum_votes).all()
assert len(df_analysis) == int(df_movies["imdb_votes"].ge(minimum_votes).sum())
assert df_analysis["movie_row_id"].is_unique
assert isinstance(family_preview, pd.DataFrame)
assert list(family_preview.columns) == ["title", "year", "rated", "imdb_rating"]
assert family_preview["rated"].isin(["PG", "PG-13"]).all()
assert family_preview["imdb_rating"].between(6, 8).all()
assert len(family_preview) <= 10
print("Exercise 5 passed!")

Exercise 5 passed!


---
## Exercise 6 — Group summaries and transformations (18 min)

### A. Country × year summary (11 min)

Create `df_summary` with one row per **`country_group` × `year`** combination from `df_analysis`. Keep missing grouping values with `dropna=False`, and return ordinary columns using `as_index=False`.

Use *named aggregation* `.agg(...)` to calculate:

| Output column | Source | Aggregation |
|:--|:--|:--|
| `n_movies` | `title` | `size` (rows, including missing values) |
| `n_with_rating` | `imdb_rating` | `count` (nonmissing only) |
| `avg_imdb` | `imdb_rating` | `mean` |
| `avg_runtime` | `runtime` | `mean` |
| `votes_total` | `imdb_votes` | `sum` |

**Question:** Why do `n_movies` and `n_with_rating` potentially differ?

**Hint:** The syntax for one named aggregation is `new_col=("old_col", "mean")`.

In [62]:
df_summary = df_analysis.groupby(["country_group", "year"], dropna=False).agg(n_movies=("title", "size"), n_with_rating=("imdb_rating", "count"), avg_imdb=("imdb_rating", "mean"), avg_runtime=("runtime", "mean"), votes_total=("imdb_votes", "sum")).reset_index() # TODO: groupby(["country_group", "year"], ...) and .agg(...)

# TODO: display the first 10 rows of df_summary
display(df_summary.head(10)) 

,country_group,year,n_movies,n_with_rating,avg_imdb,avg_runtime,votes_total
0,other,2004,1,1,6.9,98.0,24147
1,usa,1937,1,1,7.6,83.0,174379
2,usa,1940,2,2,7.6,106.5,209260
3,usa,1942,1,1,7.3,70.0,125824
4,usa,1944,1,1,6.5,71.0,11541
5,usa,1947,1,1,7.9,96.0,38715
6,usa,1949,1,1,7.0,68.0,11917
7,usa,1950,1,1,7.3,74.0,139378
8,usa,1951,1,1,7.4,75.0,123377
9,usa,1953,1,1,7.3,77.0,120707


### B. Group average on every individual movie (7 min)

Unlike `.agg(...)`, **`.transform("mean")` preserves the original number of rows**.

Create `df_deviation` as a copy of `df_analysis` and add:

- `country_mean_rating`: the IMDb mean **within the movie's country group** (`dropna=False`)
- `rating_gap`: movie's IMDb rating minus its group mean

Then display `title`, `country_group`, `imdb_rating`, `country_mean_rating`, `rating_gap` for the first 10 movies.

In [63]:
df_deviation = df_analysis.copy()
df_deviation["country_mean_rating"] = df_deviation.groupby("country_group")["imdb_rating"].transform("mean")  # TODO: groupby + transform

df_deviation["rating_gap"] = df_deviation["imdb_rating"] - df_deviation["country_mean_rating"]  # TODO

# TODO: display the 5 requested columns (first 10 rows)
display(df_deviation[["title", "country_group", "year", "imdb_rating", "rating_gap"]].head(10))

,title,country_group,year,imdb_rating,rating_gap
0,10 Things I Hate About You,usa,1999,7.3,0.470614
2,101 Dalmatians,usa_other,1996,5.7,-0.719608
4,102 Dalmatians,usa_other,2000,4.9,-1.519608
6,"20,000 Leagues Under the Sea",usa,1954,7.2,0.370614
7,A Bug's Life,usa,1998,7.2,0.370614
9,A Goofy Movie,usa_other,1995,6.8,0.380392
14,A Wrinkle in Time,usa,2018,4.2,-2.629386
15,Aladdin,usa,2019,7.0,0.170614
16,Aladdin,usa,1992,8.0,1.170614
17,Aladdin and the King of Thieves,usa,1996,6.4,-0.429386


**Self-check — Exercise 6**

In [64]:
required_summary_cols = {
    "country_group", "year", "n_movies", "n_with_rating",
    "avg_imdb", "avg_runtime", "votes_total",
}
assert isinstance(df_summary, pd.DataFrame)
assert required_summary_cols.issubset(df_summary.columns)
assert len(df_summary) == len(df_analysis.groupby(
    ["country_group", "year"], dropna=False
))
assert df_summary["n_movies"].sum() == len(df_analysis)
assert (df_summary["n_with_rating"] <= df_summary["n_movies"]).all()
assert df_summary["n_with_rating"].sum() == df_analysis["imdb_rating"].notna().sum()
assert np.isclose(df_summary["votes_total"].sum(), df_analysis["imdb_votes"].sum())
assert len(df_deviation) == len(df_analysis)
expected_mean = df_analysis.groupby("country_group", dropna=False)["imdb_rating"].transform("mean")
assert np.allclose(df_deviation["country_mean_rating"], expected_mean, equal_nan=True)
assert np.allclose(df_deviation["rating_gap"],
                   df_deviation["imdb_rating"] - expected_mean, equal_nan=True)
print("Exercise 6 passed!")

Exercise 6 passed!


---
## Exercise 7 — Wide, long, and pivot tables (16 min)

### A. Wide → long (8 min)

The source dataset has two score columns. Before combining them in one long column, bring IMDb scores to a **0–100 scale**, matching Metascores.

1. Create `df_ratings_wide` from `df_analysis` with `movie_row_id`, `title`, `year`, and `metascore`.
2. Add `imdb_score_100`.
3. Use `.melt(...)` to create `df_ratings_long` with:
   - identifier columns: `movie_row_id`, `title`, `year`
   - `rating_source`: either `imdb_score_100` or `metascore`
   - `score_100`: the rating on a 0–100 scale

**Hint:** `melt(id_vars=[...], value_vars=[...], var_name="...", value_name="...")`.

**Predict:** How many rows should the long table have relative to the wide table? Why?

In [68]:
df_ratings_wide = df_analysis[["movie_row_id", "title", "year", "metascore"]].copy()
df_ratings_wide["imdb_score_100"] = df_analysis["imdb_rating"] * 10  # TODO

df_ratings_long = df_ratings_wide.melt(id_vars=["movie_row_id", "title", "year"], value_vars=["metascore", "imdb_score_100"], var_name="rating_source", value_name="score_100")  # TODO: .melt(...)

# TODO: display the first few rows of the two tables
display(df_ratings_wide.head())
display(df_ratings_long.head())

,movie_row_id,title,year,metascore,imdb_score_100
0,0,10 Things I Hate About You,1999,70.0,73.0
2,2,101 Dalmatians,1996,49.0,57.0
4,4,102 Dalmatians,2000,35.0,49.0
6,6,"20,000 Leagues Under the Sea",1954,83.0,72.0
7,7,A Bug's Life,1998,77.0,72.0


,movie_row_id,title,year,rating_source,score_100
0,0,10 Things I Hate About You,1999,metascore,70.0
1,2,101 Dalmatians,1996,metascore,49.0
2,4,102 Dalmatians,2000,metascore,35.0
3,6,"20,000 Leagues Under the Sea",1954,metascore,83.0
4,7,A Bug's Life,1998,metascore,77.0


### B. Long → wide, and a summarising pivot table (8 min)

1. Use `.pivot(...)` to rebuild `df_ratings_back` from `df_ratings_long`. Use **`movie_row_id`, `title`, `year`** as the index, `rating_source` as columns, and `score_100` as values. Then use `.reset_index()` and remove the columns axis name with `df_ratings_back.columns.name = None`.
2. Separately create `country_year_pivot` **directly from `df_analysis`** using `.pivot_table(...)` to display **mean IMDb rating** by `country_group` (rows) and `year` (columns).

**Think:** Why does `.pivot_table()` need an aggregation function while `.pivot()` does not? When can `.pivot()` fail?

In [70]:
df_ratings_back = df_ratings_long.pivot(index=["movie_row_id", "title", "year"], columns="rating_source", values="score_100").reset_index()  # TODO: .pivot(...).reset_index()
df_ratings_back.columns.name = None  # TODO

country_year_pivot = df_analysis.pivot_table(index="country_group", columns="year", values="imdb_rating", aggfunc="mean")  # TODO: .pivot_table(..., aggfunc="mean")

# TODO: display country_year_pivot
display(country_year_pivot)

year,1937,1940,1942,1944,1947,1949,1950,1951,1953,1954,1955,1957,...,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020
country_group,,,,,,,,,,,,,,,,,,,,,,,,,
other,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
usa,7.6,7.6,7.3,6.5,7.9,7.0,7.3,7.4,7.3,7.2,7.3,7.3,...,5.95,6.88,6.64,6.95,7.225,7.475,7.3,7.6,7.183333,6.566667,7.433333,7.5
usa_other,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,6.20,6.60,7.10,7.500,NaN,NaN,6.4,NaN,7.500000,6.700000,NaN


**Self-check — Exercise 7**

In [71]:
assert isinstance(df_ratings_wide, pd.DataFrame)
assert isinstance(df_ratings_long, pd.DataFrame)
assert len(df_ratings_long) == 2 * len(df_ratings_wide)
assert set(df_ratings_long["rating_source"]) == {"imdb_score_100", "metascore"}
assert list(df_ratings_long.columns) == [
    "movie_row_id", "title", "year", "rating_source", "score_100"
]
assert np.allclose(df_ratings_wide["imdb_score_100"],
                   df_analysis["imdb_rating"] * 10, equal_nan=True)
assert isinstance(df_ratings_back, pd.DataFrame)
assert len(df_ratings_back) == len(df_ratings_wide)
a = df_ratings_back.set_index("movie_row_id").sort_index()
b = df_ratings_wide.set_index("movie_row_id").sort_index()
for col in ["metascore", "imdb_score_100"]:
    assert np.allclose(a[col], b[col], equal_nan=True)
assert isinstance(country_year_pivot, pd.DataFrame)
assert country_year_pivot.index.name == "country_group"
assert country_year_pivot.columns.name == "year"
valid_pairs = df_analysis.dropna(subset=["country_group", "year", "imdb_rating"]).groupby(
    ["country_group", "year"], observed=True
)["imdb_rating"].mean()
for (group, year), avg_rating in valid_pairs.items():
    assert np.isclose(country_year_pivot.loc[group, year], avg_rating)
print("Exercise 7 passed!")

Exercise 7 passed!


---
## Exercise 8 — Export, re-import, interpret (12 min)

### A. Save the results (7 min)

Create a local folder called `disney_pandas_practice_output` under your current working directory. Save:

- `df_clean` → `cleaned_disney.csv`
- `df_summary` → `country_year_summary.csv`

Use `index=False` in both calls. Read the summary CSV back into `summary_read_back` using `pd.read_csv(...)`. Print the file paths and the loaded table's shape.

This creates **new files only**; it does not overwrite any of the original notebook's outputs.

**Hint:** `folder.mkdir(parents=True, exist_ok=True)`; `df.to_csv(path, index=False)`; `pd.read_csv(path)`.

In [72]:
from pathlib import Path

output_dir = Path.cwd() / "disney_pandas_practice_output"
cleaned_csv = output_dir / "cleaned_disney.csv"
summary_csv = output_dir / "country_year_summary.csv"

# TODO: create output_dir
output_dir.mkdir(parents=True, exist_ok=True)

# TODO: write df_clean to cleaned_csv
df_clean.to_csv(cleaned_csv, index=False)

# TODO: write df_summary to summary_csv
df_summary.to_csv(summary_csv, index=False)

summary_read_back = pd.read_csv(summary_csv)  # TODO: read summary_csv into a DataFrame

# TODO: print output paths and summary_read_back.shape
print(cleaned_csv)
print(summary_csv)
print(summary_read_back.shape)

c:\Users\floba\OneDrive - Universität Zürich UZH\HS26\data_analysis_with_python\data-analysis-with-python-hs26\week_4\disney_pandas_practice_output\cleaned_disney.csv
c:\Users\floba\OneDrive - Universität Zürich UZH\HS26\data_analysis_with_python\data-analysis-with-python-hs26\week_4\disney_pandas_practice_output\country_year_summary.csv
(95, 7)


**Self-check — Exercise 8**

In [73]:
assert output_dir.is_dir()
assert cleaned_csv.is_file() and summary_csv.is_file()
assert isinstance(summary_read_back, pd.DataFrame)
assert summary_read_back.shape == df_summary.shape
assert set(summary_read_back.columns) == set(df_summary.columns)
assert len(pd.read_csv(cleaned_csv)) == len(df_clean)
print("Exercise 8 passed!")

Exercise 8 passed!


### B. Interpret, don't overclaim (5 min)

Write **3–5 sentences** in the markdown cell below, based on your own `df_summary` or `country_year_pivot`:

1. Give one **specific descriptive pattern** in average IMDb rating across groups or years. Name the groups/years you compare.
2. Describe exactly **which movies** are included in `df_analysis` (your denominator).
3. Explain one limitation caused by **missing ratings, country metadata, or the 10,000-vote threshold**.
4. Can these data establish that movies produced in a country are *inherently better*? Why not?

Remember: selected catalogue records are not a representative sample of all films.

**My interpretation (replace this text with your answer):**

TODO — 3–5 sentences using your actual results.

---
## Finished! A compact pandas reference (not solutions)

| Want to… | Common method |
|:--|:--|
| inspect a DataFrame | `.head()`, `.info()`, `.dtypes`, `.shape` |
| choose columns or rows | `df[[...]]`, `df.loc[mask, ...]` |
| remove rows missing a specific field | `.dropna(subset=[...])` |
| calculate values per row | `.mean(axis=1)` |
| make a new numeric or datetime column | `pd.to_numeric(...)`, `pd.to_datetime(...)` |
| get grouped statistics | `.groupby(...).agg(...)` |
| attach group statistics to existing rows | `.groupby(...).transform(...)` |
| change wide → long | `.melt(...)` |
| change long → wide | `.pivot(...)` |
| aggregate into a two-way table | `.pivot_table(...)` |
| save without an index column | `.to_csv(..., index=False)` |

**Follow-up questions to consider:** Why does `.copy()` matter after filtering? What's the difference between `size` and `count`? Why is the vote threshold a sample-selection rule?

**Note on reproducibility:** The checks focus on shapes, types and analytical relationships, not hard-coded counts from a specific Disney CSV version. If one fails, inspect the intermediate variable before moving on.